In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# =========================
# 1. Load Dataset
# =========================

df = pd.read_csv("german_credit_data.csv")

print("Shape:", df.shape)
print(df.head())

# =========================
# 2. Basic Cleaning
# =========================

# Drop unnecessary column
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Handle missing values
# Fill categorical with mode
for col in df.select_dtypes(include="object").columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fill numerical with median
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

# =========================
# 3. Identify Target
# =========================

# Kaggle dataset uses 'Risk' as target
# good -> 1, bad -> 0
df["Risk"] = df["Risk"].map({"good": 1, "bad": 0})

# =========================
# 4. Sensitive Attributes (IMPORTANT)
# =========================

# Gender is directly available
sensitive_features = ["Sex", "Age"]

# Create age groups for fairness
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 25, 60, 100],
    labels=["Young", "Adult", "Senior"]
)

# =========================
# 5. Encode Categorical Features
# =========================

categorical_cols = df.select_dtypes(include="object").columns.tolist()
categorical_cols.remove("Risk")  # exclude target

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# =========================
# 6. Feature / Target Split
# =========================

X = df_encoded.drop(columns=["Risk"])
y = df_encoded["Risk"]

# =========================
# 7. Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 8. Feature Scaling
# =========================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 9. Output Summary
# =========================

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("Sensitive Features:", sensitive_features)